# TrackMem — training on Colab

One-time setup: upload the dataset archive to Drive at `MyDrive/TrackNetDataset/TrackNetV2.zip`.

Set `MODEL` (`trackmem` or `tracknetv5`) and `EXP` (run folder name) in the first cell. Runs are saved to
`MyDrive/TrackNetV6/runs/<EXP>/`. Runtime: GPU (A100 / L4 / G4).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/TrackNetV6'           # checkpoints + logs
DATA_ZIP = '/content/drive/MyDrive/TrackNetDataset/TrackNetV2.zip'
REPO = 'panditamey/TrackMem'
MODEL = 'trackmem'                                    # trackmem | tracknetv5
EXP = 'trackmem_v2'                                   # run folder name (keeps earlier runs)
RUN = f'{DRIVE}/runs/{EXP}'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

## 1. Clone code (re-run to pull latest)

In [ ]:
import os
if os.path.isdir('/content/TrackMem/.git'):
    !git -C /content/TrackMem pull -q
else:
    !git clone -q https://github.com/{REPO}.git /content/TrackMem
%cd /content/TrackMem
!git log --oneline -1
!pip -q install -r requirements.txt

## 2. Dataset: unzip from Drive and decode frames once per session (~5-10 min)

In [ ]:
import os, shutil, zipfile
from tqdm.auto import tqdm
os.makedirs('data', exist_ok=True)
shutil.rmtree('data/TracknetV2', ignore_errors=True)
with zipfile.ZipFile(DATA_ZIP) as z:
    for m in tqdm(z.infolist(), desc='unzip', unit='file'):
        z.extract(m, 'data/')
os.rename('data/TrackNetV2', 'data/TracknetV2')
!python tools/extract_frames.py --root data/TracknetV2 --out /content/cache/frames_512x288
!du -sh /content/cache

## 3. Train
If the session disconnects, set `RESUME = True` and re-run (continues from `last.pt`).

In [ ]:
import os
BATCH, LR, WORKERS = 64, 2e-4, os.cpu_count()
RESUME = False
resume_arg = f'--resume {RUN}/last.pt' if RESUME else ''
!python train.py --model {MODEL} {resume_arg} --set experiment={EXP} \
    data.cache=/content/cache/frames_512x288 \
    train.out_dir={DRIVE}/runs \
    train.batch_size={BATCH} train.lr={LR} train.num_workers={WORKERS}

## 4. Evaluate best checkpoint on the official Test split

In [ ]:
!python evaluate.py --ckpt {RUN}/best.pt --split test --set data.cache=/content/cache/frames_512x288

## 5. Annotated video
Default: first Test rally. Set `VIDEO` to any .mp4 (e.g. on Drive) to run on your own footage.

In [ ]:
VIDEO = ''   # '' = first Test rally video
OUT = f'{DRIVE}/outputs/{MODEL}_' + (os.path.splitext(os.path.basename(VIDEO))[0] if VIDEO else 'test') + '_pred.mp4'
video_arg = f'--video "{VIDEO}"' if VIDEO else ''
!python inference.py --ckpt {RUN}/best.pt {video_arg} --out "{OUT}"

from IPython.display import Video
Video(OUT, embed=True, width=960)